# SQL Fundamentals with DuckDB (Solution)

Reference solutions for `exercise_740_sql_with_duckdb.ipynb`.

The data uses **numeric codes**:
- `Province`: 1 Western Cape, 2 Eastern Cape, 3 Northern Cape, 4 Free State,
  5 KwaZulu-Natal, 6 North West, 7 Gauteng, 8 Mpumalanga, 9 Limpopo
- `Geo_type`: 1 Urban area, 2 Tribal/traditional area, 3 Farm area
- `DERH_HSIZE`: household size (10 means "10 or more"); `DERH_HHAGE`: age of the household head

In [1]:
from pathlib import Path
import sys
import pandas as pd
import duckdb

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.append(str(project_root))

from src.utilities.project_paths import RAW_DIR

CENSUS_DIR = RAW_DIR / 'south_africa' / 'Census2022SampleSTATA'

# The households table = Census2022Households joined to Census2022Geography on QID.
# Both files are modest (~60 MB + ~26 MB) -- we never touch the 347 MB Persons file here.
# convert_categoricals=False keeps every value as a numeric code (so Geo_type = 1, etc.)
hh = pd.read_stata(
    CENSUS_DIR / 'Census2022Households.dta',
    columns=['QID', 'DERH_HSIZE', 'DERH_HHAGE'],
    convert_categoricals=False,
)
geo = pd.read_stata(
    CENSUS_DIR / 'Census2022Geography.dta',
    columns=['QID', 'Province', 'District', 'Geo_type'],
    convert_categoricals=False,
)
households = (
    hh.merge(geo, on='QID', how='left')
      [['QID', 'Province', 'District', 'DERH_HSIZE', 'DERH_HHAGE', 'Geo_type']]
)

duckdb.register('households', households)

print(f'households : {len(households):,} rows')
print('columns    :', list(households.columns))

households : 1,338,295 rows
columns    : ['QID', 'Province', 'District', 'DERH_HSIZE', 'DERH_HHAGE', 'Geo_type']


---
# Part A — Looking at the Data

| Clause | What it does |
|---|---|
| `SELECT *` | Return all columns |
| `SELECT col1, col2` | Return specific columns |
| `LIMIT n` | Return only the first n rows |

## A1. How many rows are there?

`COUNT(*)` counts every row in the table.

In [2]:
duckdb.sql("""
    SELECT COUNT(*) AS n_rows
    FROM households
""").to_df()

,n_rows
0,1338295


## A2. Peek at the data

Show the first 5 rows with all columns.

In [3]:
duckdb.sql("""
    SELECT *
    FROM households
    LIMIT 5
""").to_df()

,QID,Province,District,DERH_HSIZE,DERH_HHAGE,Geo_type
0,10000003,1,CPT,2,44,1
1,10000005,1,CPT,4,41,1
2,10000008,1,DC4,3,76,1
3,10000018,1,CPT,2,37,1
4,10000024,1,CPT,1,34,1


## A3. Select specific columns

Show only `QID`, `Province`, and `DERH_HSIZE` for the first 10 rows.

In [4]:
duckdb.sql("""
    SELECT QID, Province, DERH_HSIZE
    FROM households
    LIMIT 10
""").to_df()

,QID,Province,DERH_HSIZE
0,10000003,1,2
1,10000005,1,4
2,10000008,1,3
3,10000018,1,2
4,10000024,1,1
5,10000025,1,5
6,10000050,1,4
7,10000053,1,3
8,10000069,1,3
9,10000071,1,2


---
# Part B — Filtering with WHERE

`WHERE` keeps only rows that match a condition.

| Operator | Meaning | Example |
|---|---|---|
| `=` | equal | `Geo_type = 1` |
| `>` / `<` | greater / less than | `DERH_HSIZE > 5` |
| `>=` / `<=` | greater or equal / less or equal | `DERH_HSIZE >= 3` |
| `AND` | both conditions must be true | `Geo_type = 1 AND DERH_HSIZE > 5` |
| `OR` | at least one must be true | `Province = 7 OR Province = 8` |

## B1. Urban households only

`Geo_type = 1` means the household is in an urban area.
Show all columns, limit to 10 rows.

In [5]:
duckdb.sql("""
    SELECT *
    FROM households
    WHERE Geo_type = 1
    LIMIT 10
""").to_df()

,QID,Province,District,DERH_HSIZE,DERH_HHAGE,Geo_type
0,10000003,1,CPT,2,44,1
1,10000005,1,CPT,4,41,1
2,10000008,1,DC4,3,76,1
3,10000018,1,CPT,2,37,1
4,10000024,1,CPT,1,34,1
5,10000025,1,DC2,5,48,1
6,10000050,1,DC4,4,35,1
7,10000053,1,CPT,3,23,1
8,10000069,1,DC4,3,40,1
9,10000071,1,CPT,2,35,1


## B2. Large households

Show households with more than 6 members.
Select `QID`, `Province`, `DERH_HSIZE`.

In [6]:
duckdb.sql("""
    SELECT QID, Province, DERH_HSIZE
    FROM households
    WHERE DERH_HSIZE > 6
""").to_df()

,QID,Province,DERH_HSIZE
0,10000258,1,7
1,10000742,1,7
2,10000852,1,8
3,10001243,1,7
4,10001518,1,10
...,...,...,...
103140,91337950,9,7
103141,91337952,9,7
103142,91338061,9,7
103143,91338259,9,9


## B3. Combine two conditions

Show large households (> 6 members) in urban areas only.
Use `AND` to apply both filters at once.

In [7]:
duckdb.sql("""
    SELECT QID, Province, DERH_HSIZE, DERH_HHAGE
    FROM households
    WHERE Geo_type = 1
      AND DERH_HSIZE > 6
""").to_df()

,QID,Province,DERH_HSIZE,DERH_HHAGE
0,10000258,1,7,69
1,10000742,1,7,63
2,10000852,1,8,48
3,10001243,1,7,26
4,10001518,1,10,58
...,...,...,...,...
46613,91334430,9,10,55
46614,91334783,9,10,32
46615,91336436,9,9,62
46616,91338061,9,7,40


---
# Part C — Counting and Summarising

Aggregate functions collapse many rows into a single number.

| Function | Result |
|---|---|
| `COUNT(*)` | number of rows |
| `AVG(col)` | mean |
| `SUM(col)` | total |
| `MIN(col)` | smallest value |
| `MAX(col)` | largest value |

You can use several aggregates in a single `SELECT`.

## C1. How many urban households?

Count only the rows where `Geo_type = 1`.

In [8]:
duckdb.sql("""
    SELECT COUNT(*) AS n_urban
    FROM households
    WHERE Geo_type = 1
""").to_df()

,n_urban
0,876664


## C2. Summarise household size

Compute the average, minimum, and maximum of `DERH_HSIZE` across all rows.

In [9]:
duckdb.sql("""
    SELECT
        AVG(DERH_HSIZE) AS mean_size,
        MIN(DERH_HSIZE) AS min_size,
        MAX(DERH_HSIZE) AS max_size
    FROM households
""").to_df()

,mean_size,min_size,max_size
0,3.13025,1,10


## C3. Multiple aggregations on filtered data

For urban households: count households, mean household size, and total number of people
(the sum of household sizes).

In [10]:
duckdb.sql("""
    SELECT
        COUNT(*)        AS n_hh,
        AVG(DERH_HSIZE) AS mean_size,
        SUM(DERH_HSIZE) AS total_people
    FROM households
    WHERE Geo_type = 1
""").to_df()

,n_hh,mean_size,total_people
0,876664,2.939135,2576634.0


---
# Part D — Grouping with GROUP BY

`GROUP BY` splits rows into groups and applies aggregate functions to each group.

```sql
SELECT  group_column, AGG(value_column) AS alias
FROM    table
WHERE   condition
GROUP BY group_column
```

**Rule:** every column in `SELECT` must be either in `GROUP BY` or wrapped in an aggregate.

## D1. Count households per geo type

How many households have each value of `Geo_type` (1 urban, 2 tribal/traditional, 3 farm)?

In [11]:
duckdb.sql("""
    SELECT
        Geo_type,
        COUNT(*) AS n_hh
    FROM households
    GROUP BY Geo_type
""").to_df()

,Geo_type,n_hh
0,1,876664
1,2,411947
2,3,49684


## D2. Average household size by province

For urban households, compute the number of households and mean household size in each province.

In [12]:
duckdb.sql("""
    SELECT
        Province,
        COUNT(*)        AS n_hh,
        AVG(DERH_HSIZE) AS mean_size
    FROM households
    WHERE Geo_type = 1
    GROUP BY Province
""").to_df()

,Province,n_hh,mean_size
0,1,146698,3.013456
1,2,68592,3.027831
2,3,17650,3.391841
3,4,63280,3.147329
4,5,114454,3.001634
5,6,42269,3.054650
6,7,357437,2.781041
7,8,42671,3.054651
8,9,23613,2.998094


---
# Part E — Sorting with ORDER BY

`ORDER BY` sorts the result rows.

```sql
ORDER BY column_name        -- ascending (smallest first, this is the default)
ORDER BY column_name DESC   -- descending (largest first)
```

Combine with `LIMIT` to get the top or bottom N rows.

## E1. Provinces with the largest average household size

Repeat D2, but sort by `mean_size` descending so the largest provinces appear first.

In [13]:
duckdb.sql("""
    SELECT
        Province,
        COUNT(*)        AS n_hh,
        AVG(DERH_HSIZE) AS mean_size
    FROM households
    WHERE Geo_type = 1
    GROUP BY Province
    ORDER BY mean_size DESC
""").to_df()

,Province,n_hh,mean_size
0,3,17650,3.391841
1,4,63280,3.147329
2,8,42671,3.054651
3,6,42269,3.054650
4,2,68592,3.027831
5,1,146698,3.013456
6,5,114454,3.001634
7,9,23613,2.998094
8,7,357437,2.781041


## E2. The 5 oldest household heads

Among urban households, find the 5 whose head is oldest.
Show `QID`, `Province`, `DERH_HHAGE`.

In [14]:
duckdb.sql("""
    SELECT QID, Province, DERH_HHAGE
    FROM households
    WHERE Geo_type = 1
    ORDER BY DERH_HHAGE DESC
    LIMIT 5
""").to_df()

,QID,Province,DERH_HHAGE
0,40443006,4,110
1,20284695,2,110
2,60515130,6,109
3,40952045,4,109
4,71077851,7,109


## E3. Provinces with the fewest urban households

Sort ascending (`ASC`, the default) to find the smallest provinces first.

In [15]:
duckdb.sql("""
    SELECT
        Province,
        COUNT(*) AS n_hh
    FROM households
    WHERE Geo_type = 1
    GROUP BY Province
    ORDER BY n_hh ASC
""").to_df()

,Province,n_hh
0,3,17650
1,9,23613
2,6,42269
3,8,42671
4,4,63280
5,2,68592
6,5,114454
7,1,146698
8,7,357437
